
# Dublin Bikes: Clustering and Time Series Forecasting

## CA1 — Machine Learning for Business

This project analyses historical Dublin Bikes data to explore bike availability patterns across stations and investigate changes in bike availability over time

The project will compare two clustering algorithms and evaluate a time series forecasting model


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
data_folder = Path("../data")

In [3]:
print(data_folder.exists())

True



## 2. Data Loading

The six monthly CSV files are loaded using Pandas I will check the number of records in each file before combining the data


In [4]:

csv_files = sorted(data_folder.glob("*.csv"))

dataframes = [pd.read_csv(file) for file in csv_files]

for file, df in zip(csv_files, dataframes):
    print(file.name, "-", len(df), "rows")


dublin-bikes_AUGUST.csv - 639029 rows
dublin-bikes_JULY.csv - 601170 rows
dublin-bikes_JUNE.csv - 606948 rows
dublin-bikes_MAY.csv - 640308 rows
dublin-bikes_OCTOBER.csv - 642050 rows
dublin-bikes_SEPTEMBER.csv - 622950 rows



## 3. Data Structure

In this step, we check the column names and number of columns in each monthly file, this helps confirm whether the files have the same structure before combining them 

In [5]:

for file, df in zip(csv_files, dataframes):
    print("\n", file.name)
    print("Number of columns:", len(df.columns))
    print(list(df.columns))



 dublin-bikes_AUGUST.csv
Number of columns: 15
['system_id', 'last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'short_name', 'address', 'lat', 'lon', 'region_id', 'capacity']

 dublin-bikes_JULY.csv
Number of columns: 15
['system_id', 'last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'short_name', 'address', 'lat', 'lon', 'region_id', 'capacity']

 dublin-bikes_JUNE.csv
Number of columns: 15
['system_id', 'last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'short_name', 'address', 'lat', 'lon', 'region_id', 'capacity']

 dublin-bikes_MAY.csv
Number of columns: 15
['system_id', 'last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'short_name', 'address', 'lat', 'lon', 'reg


## 4. Combining the Monthly Data

The six monthly files have the same 15 columns I can combine them into one dataset to analyse the period from May to October 2025


In [6]:

bikes = pd.concat(dataframes, ignore_index=True)

print("Dataset shape:", bikes.shape)
print("Number of stations:", bikes["station_id"].nunique())
print("Date range:", bikes["last_reported"].min(), "to", bikes["last_reported"].max())


Dataset shape: (3752455, 15)
Number of stations: 115
Date range: 2025-05-01 00:05:00 to 2025-10-31 23:55:00



## 5. Data Quality Check

Before analysing the data, I check for missing values and duplicate rows, this helps identify data quality issues that may affect the results


In [7]:

print("Missing values per column:")
print(bikes.isnull().sum())

print("\nNumber of duplicate rows:", bikes.duplicated().sum())


Missing values per column:
system_id                    0
last_reported                0
station_id                   0
num_bikes_available          0
num_docks_available          0
is_installed                 0
is_renting                   0
is_returning                 0
name                         0
short_name             3752455
address                      0
lat                          0
lon                          0
region_id              3752455
capacity                     0
dtype: int64

Number of duplicate rows: 0



## 6. Data Consistency Check
I check whether the available bikes and available docks add up to the station capacity .. Differences may indicate inconsistent records that need further investigation


In [8]:

bikes["availability_total"] = (
    bikes["num_bikes_available"] + bikes["num_docks_available"]
)

print("Records where availability does not match capacity:")
print((bikes["availability_total"] != bikes["capacity"]).sum())


Records where availability does not match capacity:
196399


In [9]:

difference = bikes["availability_total"] - bikes["capacity"]

print("Difference summary:")
print(difference.describe())

print("\nMost common differences:")
print(difference.value_counts().head(10))


Difference summary:
count    3.752455e+06
mean    -3.774883e-01
std      3.515206e+00
min     -4.000000e+01
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.000000e+00
dtype: float64

Most common differences:
 0     3556056
-1      132487
-40      27715
-2       19685
-5        6712
-3        4954
-20       1631
-4        1203
-30       1120
-6         301
Name: count, dtype: int64


In [10]:

print(
    bikes.loc[
        difference == -40,
        ["last_reported", "station_id", "num_bikes_available",
         "num_docks_available", "capacity"]
    ].head(10)
)


            last_reported  station_id  num_bikes_available  \
29    2025-08-01 00:05:00          51                    0   
147   2025-08-01 00:15:00          51                    0   
266   2025-08-01 00:25:00          51                    0   
379   2025-08-01 00:35:00          51                    0   
497   2025-08-01 00:45:00          51                    0   
614   2025-08-01 00:55:00          51                    0   
727   2025-08-01 01:05:00          51                    0   
843   2025-08-01 01:15:00          51                    0   
956   2025-08-01 01:25:00          51                    0   
1072  2025-08-01 01:35:00          51                    0   

      num_docks_available  capacity  
29                      0        40  
147                     0        40  
266                     0        40  
379                     0        40  
497                     0        40  
614                     0        40  
727                     0        40  
843          